# 06 — Final Common Walk-Forward Evaluation

In [1]:
from pathlib import Path
import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from statsmodels.tsa.arima.model import ARIMA

try:
    from arch import arch_model
    HAS_ARCH=True
except Exception:
    HAS_ARCH=False

ROOT = Path.cwd().resolve()
if ROOT.name.lower()=="notebooks": ROOT=ROOT.parent
PROCESSED=ROOT/"data"/"processed"
TABLES=ROOT/"results"/"tables"
DIAG=ROOT/"results"/"diagnostics"
TABLES.mkdir(parents=True,exist_ok=True); DIAG.mkdir(parents=True,exist_ok=True)

HORIZONS=[5,21,63]; TEST_START=pd.Timestamp("2025-01-01")
EPS=1e-10; ANN=252; RS=42

selected=pd.read_csv(TABLES/"04_selected_ml_by_horizon.csv")
groups=pd.read_csv(TABLES/"02_feature_group_definition.csv")
display(selected)

,Horizon,Feature_Set,Transform,Model,Mean_Validation_QLIKE,Validation_QLIKE_SD,Mean_Validation_MAE,Mean_Validation_RMSE,Mean_Validation_Correlation,Total_Nonpositive_Raw_Predictions,N_Folds
0,5,F4_Market_Commodities,log,Ridge_a10,0.249683,0.019963,0.050458,0.062488,0.092123,0,2
1,21,F5_Plus_Macro,log,Ridge_a100,0.088589,0.053347,0.027826,0.035755,-0.041719,0,2
2,63,F5_Plus_Macro,raw,XGB_d2,0.078153,0.071375,0.033771,0.038660,0.276665,0,2


In [2]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [3]:
def qlike(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r=np.maximum(y*y,EPS)/np.maximum(p*p,EPS)
    return float(np.mean(r-np.log(r)-1))

def calc(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); e=p-y
    return dict(N=len(y), MAE=float(np.mean(abs(e))),
                RMSE=float(np.sqrt(np.mean(e*e))), QLIKE=qlike(y,p),
                Correlation=float(np.corrcoef(y,p)[0,1]) if len(y)>1 and np.std(p)>0 else np.nan,
                Bias=float(np.mean(e)),
                Actual_Std=float(np.std(y,ddof=1)),
                Prediction_Std=float(np.std(p,ddof=1)))

def load_h(h):
    d=pd.read_parquet(PROCESSED/f"volatility_features_h{h}.parquet").copy()
    d["Date"]=pd.to_datetime(d["Date"]); d=d.sort_values("Date").reset_index(drop=True)
    num=d.select_dtypes(include=[np.number]).columns
    d[num]=d[num].replace([np.inf,-np.inf],np.nan)
    return d

def feature_sets(d,h):
    target=f"Target_RV_{h}"
    avail=[c for c in d.columns if c not in ["Date","Sector_Return",target] and not c.startswith("Target_RV_")]
    cur=[]; out={}
    for _,r in groups.iterrows():
        add=[] if pd.isna(r["Added"]) else [x.strip() for x in str(r["Added"]).split(",") if x.strip()]
        cur=list(dict.fromkeys(cur+[x for x in add if x in avail]))
        out[r["Feature_Set"]]=cur.copy()
    return out

def make_model(name):
    if name.startswith("Ridge_a"):
        a=float(name.split("Ridge_a")[1])
        return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",Ridge(alpha=a))])
    if name.startswith("RF_leaf"):
        leaf=int(name.split("RF_leaf")[1])
        return Pipeline([("imp",SimpleImputer(strategy="median")),("m",RandomForestRegressor(n_estimators=400,min_samples_leaf=leaf,max_features=.7,n_jobs=-1,random_state=RS))])
    if name.startswith("HGB_leaf"):
        leaf=int(name.split("HGB_leaf")[1])
        return Pipeline([("imp",SimpleImputer(strategy="median")),("m",HistGradientBoostingRegressor(max_iter=250,learning_rate=.05,min_samples_leaf=leaf,random_state=RS))])
    if name.startswith("XGB_d"):
        from xgboost import XGBRegressor
        dep=int(name.split("XGB_d")[1])
        return Pipeline([("imp",SimpleImputer(strategy="median")),("m",XGBRegressor(n_estimators=400,max_depth=dep,learning_rate=.03,subsample=.8,colsample_bytree=.8,n_jobs=-1,random_state=RS,objective="reg:squarederror"))])
    raise ValueError(name)

## Static benchmarks and MASE reference

In [4]:
def static_preds(d,h):
    y=f"Target_RV_{h}"; r=d["Sector_Return"].astype(float)
    hist=np.sqrt(ANN*r.pow(2).rolling(h).mean())
    pers=np.sqrt(ANN*r.pow(2).rolling(21).mean())
    lam=.94; ev=np.full(len(d),np.nan); v=np.nan
    for i,ri in enumerate(r):
        if np.isfinite(ri):
            v=ri*ri if not np.isfinite(v) else lam*v+(1-lam)*ri*ri
        ev[i]=v
    return pd.DataFrame({"Naive_h_Target_Lag":d[y].shift(h),
                         "Historical_Matched":hist,
                         "Persistence_21D":pers,
                         "EWMA_094":np.sqrt(ANN*ev)})

## Walk-forward HAR and ARIMA

Both are evaluated at every 2025+ forecast origin.

### HAR-RV
HAR predictors are constructed directly from **observed sector returns**, avoiding any dependency on feature-column naming:

- `RV_1D`: annualised absolute daily return,
- `RV_5D`: annualised realised volatility over the latest 5 observed returns,
- `RV_21D`: annualised realised volatility over the latest 21 observed returns.

The HAR regression is trained only on target labels that have fully matured by the forecast origin (`t-h`). It is refitted at **every** test origin.

### ARIMA
ARIMA is deliberately univariate and fixed at **ARIMA(1,0,1)**. It is fitted to the safely matured target history and refitted at every test origin. Its order is not tuned using 2025+ results.

In [5]:
def har_arima(d, h):
    """
    Expanding walk-forward HAR and ARIMA(1,0,1).

    Both target-trained benchmarks obey the h-origin label embargo:
    at forecast origin t, training targets are available only through t-h.

    ARIMA convergence is explicitly audited. Repeated statsmodels
    ConvergenceWarnings are suppressed from notebook output so that numerical
    diagnostics do not overwhelm the results page.
    """
    from statsmodels.tsa.arima.model import ARIMA
    from sklearn.linear_model import LinearRegression
    from statsmodels.tools.sm_exceptions import ConvergenceWarning
    import warnings

    y = f"Target_RV_{h}"
    dates = pd.Index(sorted(d["Date"].dropna().unique()))
    har_pred, arima_pred = {}, {}
    arima_audit = []

    x = d.copy()
    r = pd.to_numeric(x["Sector_Return"], errors="coerce")
    x["HAR_RV_1D"] = np.sqrt(252.0 * r.pow(2))
    x["HAR_RV_5D"] = np.sqrt(252.0 * r.pow(2).rolling(5, min_periods=5).mean())
    x["HAR_RV_21D"] = np.sqrt(252.0 * r.pow(2).rolling(21, min_periods=21).mean())
    har_cols = ["HAR_RV_1D", "HAR_RV_5D", "HAR_RV_21D"]

    ids = x.index[(x["Date"] >= TEST_START) & x[y].notna()].tolist()

    for i in ids:
        dt = pd.Timestamp(x.loc[i, "Date"])
        pos = dates.get_loc(dt)
        if pos < h:
            continue

        latest_label_date = pd.Timestamp(dates[pos-h])

        # ---------- HAR ----------
        tr_har = x[
            (x["Date"] <= latest_label_date)
            & x[y].notna()
            & x[har_cols].notna().all(axis=1)
        ]
        if len(tr_har) >= 50 and x.loc[[i], har_cols].notna().all(axis=1).iloc[0]:
            try:
                hm = LinearRegression()
                hm.fit(tr_har[har_cols], tr_har[y])
                hp = float(hm.predict(x.loc[[i], har_cols])[0])
                har_pred[i] = max(hp, 1e-8) if np.isfinite(hp) else np.nan
            except Exception:
                har_pred[i] = np.nan
        else:
            har_pred[i] = np.nan

        # ---------- ARIMA ----------
        tr_y = (
            x.loc[(x["Date"] <= latest_label_date) & x[y].notna(), y]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .astype(float)
        )

        converged = False
        ap = np.nan
        error_type = ""

        if len(tr_y) >= 50:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", ConvergenceWarning)
                    warnings.simplefilter("ignore", UserWarning)
                    afit = ARIMA(
                        tr_y,
                        order=(1, 0, 1),
                        enforce_stationarity=False,
                        enforce_invertibility=False
                    ).fit()

                converged = bool(afit.mle_retvals.get("converged", True))
                fc = afit.forecast(steps=1)
                ap = float(np.asarray(fc).reshape(-1)[0])
                if not np.isfinite(ap):
                    ap = np.nan
                else:
                    ap = max(ap, 1e-8)

            except Exception as exc:
                error_type = type(exc).__name__
                ap = np.nan

        arima_pred[i] = ap
        arima_audit.append({
            "Date": dt,
            "Horizon": h,
            "Latest_Training_Label_Date": latest_label_date,
            "N_Train": len(tr_y),
            "Converged": converged,
            "Forecast_Available": np.isfinite(ap),
            "Error_Type": error_type
        })

    return har_pred, arima_pred, pd.DataFrame(arima_audit)


## Walk-forward GARCH(1,1)

In [6]:
def garch_preds(d, h):
    out = {}
    if not HAS_ARCH:
        return out

    y = f"Target_RV_{h}"
    ids = d.index[(d.Date >= TEST_START) & d[y].notna()].tolist()

    for i in ids:
        # Returns through t are observed at the post-ASX-close forecast origin.
        r = d.loc[:i, "Sector_Return"].dropna().to_numpy(float) * 100.0
        if len(r) < 250:
            continue

        try:
            fit = arch_model(
                r,
                mean="Zero",
                vol="GARCH",
                p=1,
                q=1,
                dist="normal",
                rescale=False
            ).fit(disp="off")

            vv = np.asarray(
                fit.forecast(horizon=h, reindex=False).variance.iloc[-1],
                float
            ) / 10000.0

            out[i] = max(
                float(np.sqrt(ANN * np.mean(vv))),
                1e-8
            )
        except Exception:
            pass

    return out

## Walk-forward selected ML

In [7]:
def ml_preds(d, h, s):
    y = f"Target_RV_{h}"
    fs = feature_sets(d, h)[s["Feature_Set"]]
    ids = d.index[(d.Date >= TEST_START) & d[y].notna()].tolist()
    out, audits = {}, []

    for i in ids:
        latest = i - h
        if latest < 1:
            continue

        tr = d.iloc[:latest + 1].dropna(subset=[y]).copy()
        model = make_model(s["Model"])
        yy = tr[y].to_numpy(float)

        if s["Transform"] == "log":
            yy = np.log(np.maximum(yy, 1e-8))

        model.fit(tr[fs], yy)

        raw = float(model.predict(d.loc[[i], fs])[0])
        pred = np.exp(raw) if s["Transform"] == "log" else raw
        out[i] = max(float(pred), 1e-8)

        audits.append({
            "Horizon": h,
            "Forecast_Date": d.loc[i, "Date"],
            "Latest_Eligible_Label_Origin": d.loc[latest, "Date"],
            "N_Train": len(tr),
            "Feature_Set": s["Feature_Set"],
            "Transform": s["Transform"],
            "Model": s["Model"],
        })

    return out, pd.DataFrame(audits)

## Generate common walk-forward forecasts

In [8]:
rows=[]; audits=[]; arima_audits=[]
for h in HORIZONS:
    print("Horizon",h)
    d=load_h(h); y=f"Target_RV_{h}"; st=static_preds(d,h)
    har,ari,arima_audit_h=har_arima(d,h); arima_audits.append(arima_audit_h); gar=garch_preds(d,h)
    s=selected.loc[selected.Horizon.eq(h)].iloc[0]
    ml,a=ml_preds(d,h,s); audits.append(a)
    ids=d.index[(d.Date>=TEST_START)&d[y].notna()].tolist()
    for i in ids:
        rows.append(dict(Date=d.loc[i,"Date"],Horizon=h,Actual=d.loc[i,y],
                         Naive_h_Target_Lag=st.loc[i,"Naive_h_Target_Lag"],
                         Historical_Matched=st.loc[i,"Historical_Matched"],
                         Persistence_21D=st.loc[i,"Persistence_21D"],
                         EWMA_094=st.loc[i,"EWMA_094"],HAR_RV=har.get(i,np.nan),
                         ARIMA_101=ari.get(i,np.nan),GARCH_11=gar.get(i,np.nan),
                         Selected_ML_WalkForward=ml.get(i,np.nan)))
pred=pd.DataFrame(rows)
audit=pd.concat(audits,ignore_index=True)
pred.to_csv(TABLES/"06_sector_walkforward_predictions_wide.csv",index=False)
audit.to_csv(DIAG/"06_direct_ml_walkforward_refit_audit.csv",index=False)
arima_audit=pd.concat(arima_audits,ignore_index=True)
arima_audit.to_csv(DIAG/"06_arima_convergence_audit.csv",index=False)
display(pred.head()); display(audit.head())
print("\nARIMA numerical-fit audit:")
display(
    arima_audit.groupby("Horizon", as_index=False).agg(
        Fits=("Converged","size"),
        Converged=("Converged","sum"),
        Convergence_Rate=("Converged","mean"),
        Forecasts_Available=("Forecast_Available","sum")
    )
)


Horizon 5
Horizon 21
Horizon 63


,Date,Horizon,Actual,Naive_h_Target_Lag,Historical_Matched,Persistence_21D,EWMA_094,HAR_RV,ARIMA_101,GARCH_11,Selected_ML_WalkForward
0,2025-01-02,5,0.148292,0.102434,0.102434,0.189976,0.178145,0.183685,0.113737,0.201768,0.159512
1,2025-01-03,5,0.178094,0.098819,0.098819,0.189861,0.172817,0.177710,0.112047,0.193295,0.152073
2,2025-01-06,5,0.142849,0.142783,0.142783,0.197155,0.177877,0.199706,0.159960,0.202867,0.145645
3,2025-01-07,5,0.157915,0.149494,0.149494,0.196995,0.174195,0.195808,0.160689,0.197082,0.145048
4,2025-01-08,5,0.132936,0.164455,0.164455,0.201579,0.175646,0.205378,0.176810,0.200225,0.150737


,Horizon,Forecast_Date,Latest_Eligible_Label_Origin,N_Train,Feature_Set,Transform,Model
0,5,2025-01-02,2024-12-23,1768,F4_Market_Commodities,log,Ridge_a10
1,5,2025-01-03,2024-12-24,1769,F4_Market_Commodities,log,Ridge_a10
2,5,2025-01-06,2024-12-27,1770,F4_Market_Commodities,log,Ridge_a10
3,5,2025-01-07,2024-12-30,1771,F4_Market_Commodities,log,Ridge_a10
4,5,2025-01-08,2024-12-31,1772,F4_Market_Commodities,log,Ridge_a10



ARIMA numerical-fit audit:


,Horizon,Fits,Converged,Convergence_Rate,Forecasts_Available
0,5,430,430,1.000000,430
1,21,414,397,0.958937,414
2,63,372,355,0.954301,372


## Common-grid final metrics

In [9]:
stock_path = TABLES / "05_stock_level_aggregated_sector_predictions.csv"
if not stock_path.exists():
    raise FileNotFoundError(
        "Missing results/tables/05_stock_level_aggregated_sector_predictions.csv. "
        "Run the corrected Notebook 05 first."
    )

stock_sector = pd.read_csv(stock_path, parse_dates=["Date"])

required_stock_cols = {"Date", "Horizon", "Stock_Level_ML_Aggregated"}
missing = required_stock_cols.difference(stock_sector.columns)
if missing:
    raise ValueError(f"Notebook 05 stock-sector output is missing columns: {sorted(missing)}")

integrated = pred.merge(
    stock_sector[["Date", "Horizon", "Stock_Level_ML_Aggregated"]],
    on=["Date", "Horizon"],
    how="inner",
    validate="one_to_one"
)

models = [
    "Naive_h_Target_Lag",
    "Historical_Matched",
    "Persistence_21D",
    "EWMA_094",
    "HAR_RV",
    "ARIMA_101",
    "GARCH_11",
    "Selected_ML_WalkForward",
    "Stock_Level_ML_Aggregated",
]
models = [m for m in models if m in integrated.columns and integrated[m].notna().any()]

metric_rows = []
common_frames = []

for h in HORIZONS:
    z = integrated[integrated["Horizon"].eq(h)].copy()
    z = z.dropna(subset=["Actual"] + models)

    if z.empty:
        raise ValueError(f"No complete common-grid observations for horizon {h}.")

    denom = float(np.mean(np.abs(z["Actual"] - z["Naive_h_Target_Lag"])))

    for model_name in models:
        q = calc(z["Actual"], z[model_name])
        q["MASE"] = q["MAE"] / denom if denom > 0 else np.nan
        q["Variability_Ratio"] = (
            q["Prediction_Std"] / q["Actual_Std"]
            if q["Actual_Std"] > 0 else np.nan
        )
        metric_rows.append({
            "Horizon": h,
            "Model": model_name,
            **q
        })

    common_frames.append(z)

final_metrics = pd.DataFrame(metric_rows)
final_common = pd.concat(common_frames, ignore_index=True)

final_metrics.to_csv(
    TABLES / "06_FINAL_common_sample_metrics.csv",
    index=False
)
final_common.to_csv(
    TABLES / "06_FINAL_common_sample_predictions.csv",
    index=False
)

print("Final common-grid observations by horizon:")
display(
    final_common.groupby("Horizon")["Date"]
    .nunique()
    .rename("N_Dates")
    .reset_index()
)

print("\nFinal model comparison:")
display(
    final_metrics.sort_values(["Horizon", "QLIKE"])[
        [
            "Horizon", "Model", "N", "QLIKE", "MAE", "RMSE",
            "MASE", "Correlation", "Bias", "Variability_Ratio"
        ]
    ]
)


Final common-grid observations by horizon:


,Horizon,N_Dates
0,5,430
1,21,414
2,63,372



Final model comparison:


,Horizon,Model,N,QLIKE,MAE,RMSE,MASE,Correlation,Bias,Variability_Ratio
6,5,GARCH_11,430,0.349235,0.084210,0.109472,0.983996,0.397021,0.031767,0.676307
4,5,HAR_RV,430,0.349481,0.078307,0.102470,0.915014,0.417447,0.010635,0.630342
8,5,Stock_Level_ML_Aggregated,430,0.380787,0.075997,0.101434,0.888021,0.385169,-0.000161,0.490922
3,5,EWMA_094,430,0.384502,0.079599,0.105857,0.930110,0.372346,0.011833,0.630458
2,5,Persistence_21D,430,0.431247,0.082476,0.112235,0.963737,0.332479,0.009820,0.731016
5,5,ARIMA_101,430,0.498846,0.084176,0.112893,0.983599,0.449782,0.004820,0.968578
7,5,Selected_ML_WalkForward,430,0.543064,0.079427,0.105869,0.928106,0.450265,-0.039918,0.355158
1,5,Historical_Matched,430,0.609748,0.085580,0.114800,1.000000,0.449485,-0.001701,1.003563
0,5,Naive_h_Target_Lag,430,0.609748,0.085580,0.114800,1.000000,0.449485,-0.001701,1.003563
17,21,Stock_Level_ML_Aggregated,414,0.184792,0.053823,0.071646,0.789663,0.471149,-0.009237,0.639931


## Primary 21-day result and reporting rule

In [10]:
primary_21 = (
    final_metrics[final_metrics["Horizon"].eq(21)]
    .sort_values("QLIKE")
    .reset_index(drop=True)
)

primary_21.to_csv(
    TABLES / "06_PRIMARY_21d_report_table.csv",
    index=False
)

display(
    primary_21[
        [
            "Model", "N", "QLIKE", "MAE", "RMSE", "MASE",
            "Correlation", "Bias", "Variability_Ratio"
        ]
    ]
)

print(
    "Primary reporting rule: interpret 21-day QLIKE first, then use "
    "MAE/RMSE/MASE, correlation, bias and variability ratio to explain "
    "the differences. Five- and 63-day results are robustness evidence."
)


,Model,N,QLIKE,MAE,RMSE,MASE,Correlation,Bias,Variability_Ratio
0,Stock_Level_ML_Aggregated,414,0.184792,0.053823,0.071646,0.789663,0.471149,-0.009237,0.639931
1,GARCH_11,414,0.204736,0.067615,0.090490,0.992000,0.371461,0.032927,0.893995
2,HAR_RV,414,0.204919,0.063859,0.084854,0.936903,0.391657,0.018146,0.892840
3,EWMA_094,414,0.244545,0.061370,0.082403,0.900380,0.395307,-0.002731,0.885193
4,ARIMA_101,414,0.308274,0.067957,0.090318,0.997023,0.366430,-0.003941,1.026066
5,Selected_ML_WalkForward,414,0.308699,0.065125,0.084995,0.955471,0.290878,-0.034568,0.509771
6,Historical_Matched,414,0.313652,0.068160,0.090534,1.000000,0.364574,-0.004600,1.027225
7,Persistence_21D,414,0.313652,0.068160,0.090534,1.000000,0.364574,-0.004600,1.027225
8,Naive_h_Target_Lag,414,0.313652,0.068160,0.090534,1.000000,0.364574,-0.004600,1.027225


Primary reporting rule: interpret 21-day QLIKE first, then use MAE/RMSE/MASE, correlation, bias and variability ratio to explain the differences. Five- and 63-day results are robustness evidence.


In [11]:
assert set(final_metrics["Horizon"]) == set(HORIZONS)
assert {"Selected_ML_WalkForward", "Stock_Level_ML_Aggregated"}.issubset(
    set(final_metrics["Model"])
)
assert (final_metrics["N"] > 0).all()
assert np.isfinite(
    final_metrics[["QLIKE", "MAE", "RMSE"]].to_numpy()
).all()

# Verify common-grid N is identical across models within each horizon.
n_check = final_metrics.groupby("Horizon")["N"].nunique()
assert (n_check == 1).all(), "Models are not being compared on identical samples."

print("Final common-sample integrity checks passed.")


Final common-sample integrity checks passed.
